In [ ]:

# 4. Notebook Robustesse et Évaluation
cells_04 = [
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": [
            "# 04 - Robustesse et Évaluation Clinique\n",
            "\n",
            "Évaluation de la robustesse et validation clinique des modèles XAI."
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "import pandas as pd\n",
            "import numpy as np\n",
            "import pickle\n",
            "import matplotlib.pyplot as plt\n",
            "import seaborn as sns\n",
            "import sys\n",
            "sys.path.append('../src')\n",
            "\n",
            "from xai_clinical.evaluation.robustness import RobustnessAnalyzer\n",
            "from xai_clinical.evaluation.clinical_validation import ClinicalValidator\n",
            "from xai_clinical.explainability.stability_analyzer import StabilityAnalyzer\n",
            "from xai_clinical.visualization.plots import plot_robustness_dashboard\n",
            "\n",
            "import joblib\n",
            "import warnings\n",
            "warnings.filterwarnings('ignore')"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 1. Chargement"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Charger données et modèles\n",
            "with open('../data/processed/preprocessed_data.pkl', 'rb') as f:\n",
            "    data = pickle.load(f)\n",
            "\n",
            "X_test = data['X_test']\n",
            "y_test = data['y_test']\n",
            "feature_names = data['feature_names']\n",
            "\n",
            "# Charger modèle\n",
            "model = joblib.load('../models/saved_models/random_forest.pkl')\n",
            "\n",
            "print(f\"Test set: {X_test.shape}\")"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 2. Analyse de Robustesse"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Robustness analyzer\n",
            "robustness_analyzer = RobustnessAnalyzer(model, feature_names)\n",
            "\n",
            "# Test avec bruit\n",
            "noise_levels = [0.01, 0.05, 0.1, 0.15, 0.2]\n",
            "robustness_results = robustness_analyzer.test_noise_robustness(\n",
            "    X_test, y_test, noise_levels\n",
            ")\n",
            "\n",
            "# Visualiser\n",
            "robustness_analyzer.plot_robustness_curve(robustness_results)"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Test avec données manquantes\n",
            "missing_rates = [0.05, 0.1, 0.15, 0.2, 0.25]\n",
            "missing_results = robustness_analyzer.test_missing_data_robustness(\n",
            "    X_test, y_test, missing_rates\n",
            ")\n",
            "\n",
            "robustness_analyzer.plot_missing_data_impact(missing_results)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 3. Stabilité des Explications"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Stability analyzer\n",
            "stability_analyzer = StabilityAnalyzer(model, feature_names)\n",
            "\n",
            "# Test stabilité SHAP\n",
            "stability_scores = stability_analyzer.compute_shap_stability(\n",
            "    X_test[:50], n_perturbations=10, noise_level=0.01\n",
            ")\n",
            "\n",
            "print(f\"Stabilité moyenne SHAP: {np.mean(stability_scores):.3f}\")\n",
            "print(f\"Écart-type: {np.std(stability_scores):.3f}\")"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Consistance across similar patients\n",
            "consistency_scores = stability_analyzer.compute_explanation_consistency(\n",
            "    X_test[:50], threshold=0.1\n",
            ")\n",
            "\n",
            "plt.figure(figsize=(10, 6))\n",
            "plt.hist(consistency_scores, bins=20, color='skyblue', edgecolor='black')\n",
            "plt.xlabel('Consistency Score')\n",
            "plt.ylabel('Frequency')\n",
            "plt.title('Distribution de la Consistance des Explications')\n",
            "plt.axvline(np.mean(consistency_scores), color='red', linestyle='--', label='Mean')\n",
            "plt.legend()\n",
            "plt.show()"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 4. Validation Clinique"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Clinical validator\n",
            "clinical_validator = ClinicalValidator(model, feature_names)\n",
            "\n",
            "# Decision curve analysis\n",
            "y_prob = model.predict_proba(X_test)[:, 1]\n",
            "dca_results = clinical_validator.decision_curve_analysis(y_test, y_prob)\n",
            "\n",
            "clinical_validator.plot_decision_curve(dca_results)"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Calibration analysis\n",
            "calibration_results = clinical_validator.calibration_analysis(y_test, y_prob)\n",
            "\n",
            "clinical_validator.plot_calibration_curve(calibration_results)"
        ]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Risk stratification\n",
            "risk_strata = clinical_validator.stratify_risk(y_prob, n_strata=3)\n",
            "stratification_table = clinical_validator.evaluate_risk_stratification(\n",
            "    y_test, y_prob, risk_strata\n",
            ")\n",
            "\n",
            "print(\"Table de Stratification du Risque:\")\n",
            "print(stratification_table)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 5. Fairness et Biais"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Simuler variable sensible (âge > 65)\n",
            "# Dans un vrai cas, utiliser données démographiques\n",
            "np.random.seed(42)\n",
            "sensitive_attr = np.random.choice(['<65', '>=65'], size=len(y_test))\n",
            "\n",
            "# Fairness analysis\n",
            "fairness_results = clinical_validator.fairness_analysis(\n",
            "    y_test, y_prob, sensitive_attr\n",
            ")\n",
            "\n",
            "clinical_validator.plot_fairness_metrics(fairness_results)"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 6. Dashboard de Robustesse"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Dashboard complet\n",
            "fig = plot_robustness_dashboard(\n",
            "    robustness_results,\n",
            "    stability_scores,\n",
            "    calibration_results,\n",
            "    fairness_results\n",
            ")\n",
            "\n",
            "plt.savefig('../reports/figures/robustness_dashboard.png', dpi=300, bbox_inches='tight')\n",
            "plt.show()\n",
            "print('Dashboard sauvegardé')"
        ]
    },
    {
        "cell_type": "markdown",
        "metadata": {},
        "source": ["## 7. Rapport Final"]
    },
    {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": [
            "# Générer rapport\n",
            "report = clinical_validator.generate_clinical_report(\n",
            "    y_test, y_prob, robustness_results, stability_scores\n",
            ")\n",
            "\n",
            "with open('../reports/rapport_final.md', 'w') as f:\n",
            "    f.write(report)\n",
            "\n",
            "print('Rapport clinique généré: ../reports/rapport_final.md')"
        ]
    }
]

notebook_04 = notebook_structure.copy()
notebook_04["cells"] = cells_04

with open(f"{base_path}/notebooks/04_robustesse_evaluation.ipynb", "w") as f:
    json.dump(notebook_04, f, indent=1)



